In [ ]:
from pathlib import Path
import hashlib,json,os,signal,subprocess,sys,tarfile,time,traceback,urllib.request,zipfile
import psutil
RUN_ID='__RUN_ID__'
WORK=Path('/kaggle/working')/'runs'/'b321'/RUN_ID;WORK.mkdir(parents=True,exist_ok=False)
TEMP=Path('/kaggle/tmp')/RUN_ID;TEMP.mkdir(parents=True,exist_ok=False)
RECEIPTS=WORK/'receipts';RECEIPTS.mkdir()
SETUP=WORK/'logs'/'setup';SETUP.mkdir(parents=True,exist_ok=False)
START=time.monotonic();DEADLINE=START+1800
status={'status':'SETUP','source_sha256':'47d41588c488dd9668c5ac5747f8ffbddd80df5f7d9b0a016ce709062f71e4b1','arms':['er_uniform','a0_tail_ctx'],'scientific_acceptance':False}
def save():
    (RECEIPTS/'RUN_STATUS.json').write_text(json.dumps(status,indent=2)+'\n',encoding='utf8')
def registered_resource_stop(exit_code,result):
    if exit_code!=2 or not isinstance(result,dict):return False
    reason=result.get('status')
    if result.get('experiment')!='B321' or result.get('studies')!={}:return False
    pf=result.get('preflight',{})
    if not isinstance(pf,dict) or pf.get('status')!=reason:return False
    def positive(x):return not isinstance(x,bool) and isinstance(x,(int,float)) and 0<x<float('inf')
    if reason in ('PREFLIGHT_INSUFFICIENT_MEMORY','PREFLIGHT_INSUFFICIENT_HOST_MEMORY'):
        memory=pf.get('memory',{})
        if not isinstance(memory,dict) or memory.get('limit_bytes')!=16*1024**3 or memory.get('worker_limit_bytes')!=8*1024**3:return False
        projected=memory.get('projected_group_RSS_bytes');workers=memory.get('projected_worker_RSS_bytes',{})
        if not positive(projected) or set(workers)!={'er_uniform','a0_tail_ctx'} or not all(positive(v) for v in workers.values()):return False
        if reason=='PREFLIGHT_INSUFFICIENT_MEMORY':return projected>16*1024**3 or max(workers.values())>8*1024**3
        required=memory.get('host_required_bytes');available=memory.get('host',{}).get('effective_available_bytes')
        return positive(required) and positive(available) and required>available
    if reason!='PREFLIGHT_INSUFFICIENT_CAP':return False
    projection=pf.get('projection',{})
    if not isinstance(projection,dict):return False
    checks=[(projection.get('remaining_worker_with_margin'),projection.get('remaining_worker_cap')),(projection.get('remaining_with_margin_export'),projection.get('remaining_whole_cap'))]
    return all(positive(estimate) and positive(remaining) for estimate,remaining in checks) and any(estimate>remaining for estimate,remaining in checks)

def command(args,name,seconds,ram=False,allow_resource_stop=False):
    limit=min(DEADLINE-120,time.monotonic()+seconds)
    cpu_used=0.0;seen_cpu={}
    print('COMMAND_START',name,flush=True)
    env=os.environ.copy();env.update(PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1',PYTHONPATH=str(ROOT/'src'),CUBLAS_WORKSPACE_CONFIG=':4096:8',XFORMERS_DISABLED='1',OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',NRCAD_LOCAL_ROOT=str(TEMP/'local'),NRCAD_RUN_DEADLINE_MONOTONIC=str(DEADLINE))
    log=SETUP/(name+'.log')
    with log.open('x',encoding='utf8') as output,log.open(encoding='utf8') as reader:
        proc=subprocess.Popen([str(x) for x in args],cwd=ROOT,env=env,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                text=reader.read()
                if text:print(text,end='',flush=True)
                if time.monotonic()>limit:raise TimeoutError(name+' cap exceeded')
                if ram:
                    try:
                        parent=psutil.Process(proc.pid);rss=sum(p.memory_info().rss for p in [parent,*parent.children(recursive=True)] if p.is_running())
                        if rss>16*1024**3:raise MemoryError(name+' process-group RSS cap16GiB exceeded')
                        for child in [parent,*parent.children(recursive=True)]:
                            try:
                                ct=child.cpu_times();seen_cpu[(child.pid,child.create_time())]=ct.user+ct.system
                            except psutil.Error:pass
                        cpu_used=sum(seen_cpu.values())
                        if cpu_used>7200:raise TimeoutError('Aggregate CPU cap7200s exceeded')
                        # Same race-safe census as the parent, with root source imported only after verification.
                        import importlib.util
                        spec=importlib.util.spec_from_file_location('b321_disk_census',ROOT/'src/nested_replay_cad/storage.py')
                        disk_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(disk_module)
                        used=disk_module.census(WORK)
                        if used>8*1024**3:raise MemoryError('Output cap8GiB exceeded')
                    except psutil.NoSuchProcess:pass
                time.sleep(1)
            text=reader.read()
            if text:print(text,end='',flush=True)
            if proc.returncode:
                payload_path=WORK/'outputs/result.json'
                if allow_resource_stop and payload_path.is_file():
                    payload=json.loads(payload_path.read_text(encoding='utf8'))
                    if registered_resource_stop(proc.returncode,payload):
                        status['status']='BLOCKED_RESOURCE_PREFLIGHT'
                        status['worker_status']=payload['status']
                        print('REGISTERED_STOP',payload['status'],'full training not started.',flush=True)
                        return False
                raise RuntimeError(name+' exit='+str(proc.returncode))
        finally:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
                try:proc.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()
    print('COMMAND_DONE',name,flush=True)
    return True
def load_source(input_root,destination,zip_sha,manifest_sha):
    import shutil
    candidates=sorted(input_root.rglob('source.zip'))
    valid=[p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest()==zip_sha]
    if destination.exists():raise FileExistsError('Source destination already exists')
    destination.mkdir(parents=True)
    if valid:
        with zipfile.ZipFile(valid[0]) as archive:
            for info in archive.infolist():
                if not (destination/info.filename).resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe source path')
            archive.extractall(destination)
        layout='zip'
    else:
        bundles=sorted(input_root.rglob('BUNDLE.json'))
        valid_bundles=[p for p in bundles if hashlib.sha256(p.read_bytes()).hexdigest()==manifest_sha]
        if not valid_bundles:
            raise ValueError('No pinned source found: ZIP candidates='+str(len(candidates))+'; expanded manifests='+str(len(bundles)))
        bundle=valid_bundles[0];manifest=json.loads(bundle.read_bytes())
        for name,expected in manifest.items():
            target=destination/name
            if not target.resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe manifest path')
            source=bundle.parent/name
            if hashlib.sha256(source.read_bytes()).hexdigest()!=expected:raise ValueError('Mounted source mismatch '+name)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,target)
        shutil.copyfile(bundle,destination/'BUNDLE.json');layout='expanded'
    raw=(destination/'BUNDLE.json').read_bytes()
    if hashlib.sha256(raw).hexdigest()!=manifest_sha:raise ValueError('Pinned source manifest mismatch')
    manifest=json.loads(raw)
    for name,expected in manifest.items():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe manifest path')
        if hashlib.sha256((destination/name).read_bytes()).hexdigest()!=expected:raise ValueError('Source member mismatch '+name)
    return manifest,layout

try:
    save();print('B321_BUDGET_BOOTSTRAP',json.dumps(status),flush=True)
    ROOT=TEMP/'source'
    bundle,layout=load_source(Path('/kaggle/input'),ROOT,'47d41588c488dd9668c5ac5747f8ffbddd80df5f7d9b0a016ce709062f71e4b1','003b91932dbd6713782f0fd90f64dbdedcea12ab53923fe6a31e2a8dcc06f2f5')
    status['source_layout']=layout
    print('SOURCE_LAYOUT',layout,flush=True)
    print('VERIFIED_SOURCE',len(bundle),'files',flush=True)
    import shutil
    package=WORK/'package';package.mkdir()
    archive_candidates=[p for p in Path('/kaggle/input').rglob('source.zip') if hashlib.sha256(p.read_bytes()).hexdigest()==status['source_sha256']]
    if archive_candidates:shutil.copyfile(archive_candidates[0],package/'source.zip')
    else:
        with zipfile.ZipFile(package/'source.zip','x',zipfile.ZIP_DEFLATED) as archive:
            for name in bundle:archive.write(ROOT/name,name)
            archive.write(ROOT/'BUNDLE.json','BUNDLE.json')
    (RECEIPTS/'source_manifest.json').write_text(json.dumps(bundle,indent=2)+'\n',encoding='utf8')
    pixi_tar=TEMP/'pixi.tar.gz'
    urllib.request.urlretrieve('https://github.com/prefix-dev/pixi/releases/download/v0.69.0/pixi-x86_64-unknown-linux-musl.tar.gz',pixi_tar)
    bin_dir=TEMP/'bin';bin_dir.mkdir()
    with tarfile.open(pixi_tar) as archive:archive.extractall(bin_dir,filter='data')
    PIXI=bin_dir/'pixi';PIXI.chmod(0o755)
    command([PIXI,'--version'],'pixi-version',60)
    command([PIXI,'install','--locked','-e','cad'],'install',900)
    python=ROOT/'.pixi/envs/cad/bin/python'
    command([python,'-I','-S','-B','tests/test_storage_race.py'],'filesystem-race',120,True)
    completed=command([python,'-B','scripts/qualify_budget_resume.py','--output',WORK/'outputs'],'native-checkpoint-qualification',1200,True)
    if completed:status['status']='COMPLETE_REVIEW_PENDING'
except BaseException as error:
    status.update(status='FAILED',error=repr(error),traceback=traceback.format_exc());raise
finally:
    status['seconds']=time.monotonic()-START;save();print('B321_TERMINAL',json.dumps(status),flush=True)
